## Build tables from the images

In [19]:
import sys
sys.path.append("../")
import numpy as np

from pathlib import Path
from feature_engineering.build_tables_extended import collect_items_extended_multilabel, build_split_table_tif_multilabel_extended

from evaluation.generate_test import spatial_train_test_split

from config import ROOT_NAME, TRAIN_FILES, LABEL_PATHS, TEST_FILES_TEMP, \
                   FEATURE_COLS_BASELINE, FEATURE_COLS_BASELINE_EXTENDED, \
                   TARGET_COLS2020, TARGET_COLS2021

PROJECT_ROOT = Path("../").resolve()
ROOT = (PROJECT_ROOT / ROOT_NAME).resolve()

In [2]:
# feature extraction v2 (5 bands usage) + 2020 and 2021 labels
train_items = collect_items_extended_multilabel(
    root=ROOT,
    folder_names=TRAIN_FILES,
    label_paths=LABEL_PATHS,
)

dataset = build_split_table_tif_multilabel_extended(
    items=train_items,
    kernel_size=101,
    keep_borders=True,   
    stride=101,
)

Processing urn:eop:VITO:TERRASCOPE_S2_TOC_V2:S2B_20200327T101629_32UPV_TOC_V210__20200327T101629 ...
Processing urn:eop:VITO:TERRASCOPE_S2_TOC_V2:S2A_20200809T102031_32UPV_TOC_V210__20200809T102031 ...
Processing urn:eop:VITO:TERRASCOPE_S2_TOC_V2:S2A_20200918T102031_32UPV_TOC_V210__20200918T102031 ...


In [4]:
# generate 1 for temporal test (from 2021 winter)
items_test_temp = collect_items_extended_multilabel(
    root=ROOT,
    folder_names=TEST_FILES_TEMP,
    label_paths=LABEL_PATHS,
)

test_temp = build_split_table_tif_multilabel_extended(
            items=items_test_temp,
            kernel_size=101,
            keep_borders=True,
            stride=101,
)

Processing urn:eop:VITO:TERRASCOPE_S2_TOC_V2:S2B_20210220T101939_32UPV_TOC_V210__20210220T101939 ...
Processing urn:eop:VITO:TERRASCOPE_S2_TOC_V2:S2A_20211222T102441_32UPV_TOC_V210__20211222T102441 ...


### Make train/test split with 2 different hold-out types

In [5]:
train_coors_df, test_coors_df = spatial_train_test_split(dataset, "x", "y")
train_df = dataset.merge(train_coors_df, on=["x", "y"], how="inner")

# same dates, diff xy
test_spatial = dataset.merge(test_coors_df, on=["x", "y"], how="inner")

# same xy, diff dates
test_temp_only = test_temp.merge(train_coors_df, on=["x", "y"], how="inner")

# diff xy, diff dates
test_spatial_temp = test_temp.merge(test_coors_df, on=["x", "y"], how="inner")

In [6]:
# even too much for some of them lol
train_df.shape, test_spatial.shape, test_temp_only.shape, test_spatial_temp.shape

((50316, 51), (12579, 51), (33544, 51), (8386, 51))

## Train baseline

In [21]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import classification_report, mean_absolute_error, mean_squared_error
import os
import random

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [22]:
CLASS_NAMES = ['urban', 'water', 'vegetation']

In [23]:
def normalize_predictions_to_simplex(y_pred: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    Clip predictions to [0, +inf) and renormalize rows to sum to 1.
    """
    y_pred = np.asarray(y_pred, dtype=np.float64)
    y_pred = np.clip(y_pred, 0.0, None)

    row_sums = y_pred.sum(axis=1, keepdims=True)
    zero_rows = row_sums.squeeze(-1) < eps

    # fallback: uniform distribution if model predicted all zeros
    if np.any(zero_rows):
        y_pred[zero_rows] = 1.0
        row_sums = y_pred.sum(axis=1, keepdims=True)

    y_pred = y_pred / row_sums
    return y_pred


def dominant_class_from_distribution(y: np.ndarray) -> np.ndarray:
    """
    Convert 3-probability target/prediction into hard class via argmax.
    """
    return np.argmax(y, axis=1)


def fit_and_predict_distribution_model(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    target_cols: list[str],
    alpha: float = 1.0,
):
    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_cols].to_numpy(dtype=np.float64)

    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_cols].to_numpy(dtype=np.float64)

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("regressor", Ridge(alpha=alpha, random_state=42)),
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred = normalize_predictions_to_simplex(y_pred)

    return model, y_test, y_pred


def evaluate_distribution_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: list[str],
    digits: int = 4,
):
    """
    Returns both regression-style metrics and classification_report
    computed on dominant class (argmax).
    """
    metrics = {
        "mae_macro": float(mean_absolute_error(y_true, y_pred)),
        "rmse_macro": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae_per_target": {
            class_names[i]: float(mean_absolute_error(y_true[:, i], y_pred[:, i]))
            for i in range(len(class_names))
        },
        "rmse_per_target": {
            class_names[i]: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
            for i in range(len(class_names))
        },
    }

    y_true_cls = dominant_class_from_distribution(y_true)
    y_pred_cls = dominant_class_from_distribution(y_pred)

    clf_report_dict = classification_report(
        y_true_cls,
        y_pred_cls,
        target_names=class_names,
        digits=digits,
        output_dict=True,
        zero_division=0,
    )

    clf_report_text = classification_report(
        y_true_cls,
        y_pred_cls,
        target_names=class_names,
        digits=digits,
        zero_division=0,
    )

    return {
        "metrics": metrics,
        "classification_report_dict": clf_report_dict,
        "classification_report_text": clf_report_text,
    }


def run_experiment_suite(
    train_df: pd.DataFrame,
    test_sets: dict[str, pd.DataFrame],
    feature_sets: dict[str, list[str]],
    target_cols: list[str],
    class_names: list[str],
    alpha: float = 1.0,
):
    """
    Train one model per feature set on train_df.
    Evaluate on all provided test sets.
    """
    all_results = []

    for feature_set_name, feature_cols in feature_sets.items():
        print("=" * 100)
        print(f"FEATURE SET: {feature_set_name}")
        print(f"N features: {len(feature_cols)}")

        for test_name, test_df in test_sets.items():
            model, y_true, y_pred = fit_and_predict_distribution_model(
                train_df=train_df,
                test_df=test_df,
                feature_cols=feature_cols,
                target_cols=target_cols,
                alpha=alpha,
            )

            result = evaluate_distribution_predictions(
                y_true=y_true,
                y_pred=y_pred,
                class_names=class_names,
            )

            print("-" * 100)
            print(f"Test set: {test_name}")
            print("Regression metrics:")
            print(result["metrics"])
            print()
            print("classification_report (dominant class via argmax):")
            print(result["classification_report_text"])

            all_results.append({
                "feature_set": feature_set_name,
                "test_set": test_name,
                "n_features": len(feature_cols),
                "mae_macro": result["metrics"]["mae_macro"],
                "rmse_macro": result["metrics"]["rmse_macro"],
                "macro_f1": result["classification_report_dict"]["macro avg"]["f1-score"],
                "weighted_f1": result["classification_report_dict"]["weighted avg"]["f1-score"],
                "accuracy": result["classification_report_dict"]["accuracy"],
            })

    return pd.DataFrame(all_results)

In [27]:
feature_sets = {
    "baseline": FEATURE_COLS_BASELINE,
    "baseline_extended": FEATURE_COLS_BASELINE_EXTENDED,
}

test_sets = {
    "test_spatial": test_spatial,
    "test_temporal_only": test_temp_only,
    "test_spatial_temporal": test_spatial_temp,
}

results_df = run_experiment_suite(
    train_df=train_df,
    test_sets=test_sets,
    feature_sets=feature_sets,
    target_cols=TARGET_COLS2020,
    class_names=CLASS_NAMES,
    alpha=1.0,
)

results_df

FEATURE SET: baseline
N features: 35
----------------------------------------------------------------------------------------------------
Test set: test_spatial
Regression metrics:
{'mae_macro': 0.03911052533802715, 'rmse_macro': 0.0748623848088456, 'mae_per_target': {'urban': 0.04951320543964091, 'water': 0.012325952052592872, 'vegetation': 0.05549241852184713}, 'rmse_per_target': {'urban': 0.08652521759213547, 'water': 0.03255412565010989, 'vegetation': 0.09092164539642651}}

classification_report (dominant class via argmax):
              precision    recall  f1-score   support

       urban     0.0000    0.0000    0.0000       252
       water     0.0000    0.0000    0.0000        21
  vegetation     0.9783    1.0000    0.9890     12306

    accuracy                         0.9783     12579
   macro avg     0.3261    0.3333    0.3297     12579
weighted avg     0.9571    0.9783    0.9676     12579

-------------------------------------------------------------------------------------

,feature_set,test_set,n_features,mae_macro,rmse_macro,macro_f1,weighted_f1,accuracy
0,baseline,test_spatial,35,0.039111,0.074862,0.329677,0.967565,0.978297
1,baseline,test_temporal_only,35,0.084898,0.118714,0.330158,0.971813,0.981129
2,baseline,test_spatial_temporal,35,0.085473,0.119882,0.329677,0.967565,0.978297
3,baseline_extended,test_spatial,41,0.037202,0.070678,0.414451,0.968534,0.978536
4,baseline_extended,test_temporal_only,41,0.097767,0.129832,0.505920,0.974298,0.981666
5,baseline_extended,test_spatial_temporal,41,0.097292,0.129880,0.468629,0.969189,0.977820
